<a href="https://colab.research.google.com/github/Ekrem-Guler/RAG-Based-Mathematical-Problem-Solver/blob/main/RAG_MATH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install chromadb

In [ ]:
import pandas as pd
import numpy as np
import chromadb
from datasets import load_dataset

In [ ]:
ds = load_dataset("EleutherAI/hendrycks_math", "counting_and_probability")
ds

In [ ]:
ds_orca = load_dataset("microsoft/orca-math-word-problems-200k")

In [ ]:
ds_geo = load_dataset("EleutherAI/hendrycks_math", "geometry")
ds_geo

In [ ]:
ds_alg = load_dataset("EleutherAI/hendrycks_math", "algebra")
ds_alg

In [ ]:
data_train = pd.read_csv("train.csv")
data_test = pd.read_csv("test.csv")
data1 =pd.concat([data_train,data_test])

In [ ]:
ds8= pd.DataFrame.from_dict(ds_orca["train"])
ds8.head()

In [ ]:
ds8["problem"] = ds8["question"]
ds8["solution"]= ds8['answer']
ds8 = ds8.drop(columns=["question","answer"])
ds8.isna().sum()

In [ ]:
ds1= pd.DataFrame.from_dict(ds["train"])
ds2 = pd.DataFrame.from_dict(ds_geo["train"])
ds3 = pd.DataFrame.from_dict(ds_alg["train"])
ds4 = pd.DataFrame.from_dict(ds["test"])
ds5 = pd.DataFrame.from_dict(ds_geo["test"])
ds6 = pd.DataFrame.from_dict(ds_alg["test"])
ds7 = pd.concat([ds1,ds2,ds3,ds4,ds5,ds6,data1,ds8],ignore_index=True)
ds7

In [ ]:
ds7.to_csv("out.csv", index=False)

In [ ]:
sentences = ds7["problem"].to_list()
sentences_solution = ds7["solution"].to_list()
print(len(sentences_solution))

In [ ]:
#pip install mathematics_dataset

In [ ]:
#import mathematics_dataset

In [ ]:
#mathematics_dataset

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(sentences)
print(len(embeddings[0]))

In [ ]:
chroma_client = chromadb.Client()
collection_math = chroma_client.create_collection(name="math_collection1",configuration={"hnsw": {"space": "cosine"}})

In [ ]:
def sentence_numbers(left,right,stop):
    collection_math.add(
      documents=sentences[left:right],
      ids = [str(i+left) for i in range(right-left)],
      metadatas=[{"source": s} for s in sentences_solution[left:right]],
      embeddings=embeddings[left:right].tolist()
    )
    if(right<stop):
        left = right
        if(right+5461<stop):
            right = right+5461
        else:
            right = stop
        sentence_numbers(left,right,stop)

In [ ]:
left = 0
right = 5461
stop = len(sentences_solution)
sentence_numbers(left,right,stop)

In [ ]:
collection_math.add(
    documents=sentences[0:5461],
    ids = [str(i) for i in range(5461)],
    metadatas=[{"source": s} for s in sentences_solution[0:5461]],
    embeddings=embeddings[0:5461].tolist()
)


In [ ]:
collection_math.add(
    documents=sentences[5461:10922],
    ids = [str(i+5461) for i in range(5461)],
    metadatas=[{"source": s} for s in sentences_solution[5461:10922]],
    embeddings=embeddings[5461:10922].tolist()
)


In [ ]:
collection_math.add(
    documents=sentences[10922:16383],
    ids = [str(i+10922) for i in range(5461)],
    metadatas=[{"source": s} for s in sentences_solution[10922:16383]],
    embeddings=embeddings[10922:16383].tolist()
)


In [ ]:
collection_math.add(
    documents=sentences[16383:18023],
    ids = [str(i+16383) for i in range(1640)],
    metadatas=[{"source": s} for s in sentences_solution[16383:18023]],
    embeddings=embeddings[16383:18023].tolist()
)


In [ ]:

results = collection_math.query(
    query_texts=[
         "If every positive integer can be colored in one of k colors such that any two numbers whose difference or ratio is 2 are different colors, what is the smallest possible value of k?"
    ],
    n_results=10,
    include=["embeddings", "documents", "distances","metadatas"]
)

retrieved_solution = results["metadatas"][0][0]["source"]
retrieved_question = results["documents"][0][0]
print(f"Retrieved Solution: {retrieved_solution}\n Retrieved Question: {retrieved_question}")
for i in range(len(results['ids'][0])):
    print(f"ID: {results['ids'][0][i]}")
    print(f"Distance: {results['distances'][0][i]}")
    print(f"Content snippet: {results['documents'][0][i][:50]}...")

In [ ]:
from huggingface_hub import InferenceClient


prompt = f"""Use the following RAG system
Retrieved Question: {retrieved_question},
Retrieved Solution: {retrieved_solution},
Question: If every positive integer can be colored in one of k colors such that any two numbers whose difference or ratio is 2 are different colors, what is the smallest possible value of k?
Solution:
"""
client = InferenceClient()

completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
)
print(completion.choices[0].message.content)